## Quantum State Tomography with JuMP

Quantum state tomography estimates a density matrix from measurement data. This is an optimization problem with physics constraints:

- the state should explain the observed measurement probabilities;
- the trace should be one, because total probability is normalized;
- the density matrix should be positive semidefinite, because all measurement probabilities must be nonnegative.

We use a two-level quantum system so that the density matrix is a real Hermitian `2 × 2` matrix in the core notebook:

$$
\rho = \begin{bmatrix}
\rho_{11} & \rho_{12} \\
\rho_{12} & \rho_{22}
\end{bmatrix}.
$$

The Born rule gives the probability $p_i$ of measuring $E_i$,

$$
p_i = \mathrm{tr}(E_i\rho).
$$

This version intentionally leaves selected implementations as `TODO` exercises. Replace each `error("TODO: ...")` line with working Julia/JuMP code.


**Attribution:** Licensed under <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC-SA 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a></p>

```text
Andy Goldschmidt 
Andy.Goldschmidt@jhuapl.edu
Johns Hopkins Applied Physics Laboratory
```

## Setup

Run these cells from the `lecture-1-optimization` environment:

```julia
julia --project=.
```


In [146]:
using CairoMakie
using Ipopt
using JuMP
using LinearAlgebra
using DataFrames
using Random
using GLMakie
using QuantumToolbox

## Problem data and the Born rule

We start with four rank-one projective measurements: the computational basis states $|0\rangle$, $|1\rangle$ and the $X$-basis states $|+\rangle$, $|-\rangle$.

The first core exercise is to compute the model-predicted probabilities using the Born rule. The helper `predicted_probability(E, ρ)` should work both when `ρ` is an ordinary numeric matrix and when `ρ` is a matrix of JuMP decision variables.


In [195]:
states = Dict(
    "|0⟩" => [1.0, 0.0],
    "|1⟩" => [0.0, 1.0],
    "|+⟩" => normalize([1.0, 1.0]),
    "|−⟩" => normalize([1.0, -1.0]),
)

# Separating the lookup table from the desired measurements
measurement_names = ["|0⟩", "|1⟩", "|+⟩", "|−⟩"]

projector(ψ) = ψ * ψ'

# Example: |0⟩⟨0|
projector(states["|−⟩"])
measurements = [projector(states[name]) for name in measurement_names]


4-element Vector{Matrix{Float64}}:
 [1.0 0.0; 0.0 0.0]
 [0.0 0.0; 0.0 1.0]
 [0.4999999999999999 0.4999999999999999; 0.4999999999999999 0.4999999999999999]
 [0.4999999999999999 -0.4999999999999999; -0.4999999999999999 0.4999999999999999]

In [ ]:
# function basic_reader(E, ρ)
#     prob0 = ρ[1,1]
#     prob1 = ρ[2,2]

#     if E == "|0⟩"
#         return prob0
#     end
#     if E == "|1⟩"
#         return prob1
#     end

# end

function predicted_probability(E, ρ)
    Trace = 0
    n = size(ρ)[1]
    prob_init = E*ρ
    for i in range(1,n)
        for j in range(1,n)
            if i ==j 
                Trace += prob_init[i,j]
            end
        end
    end
    return Trace
end

predicted_probability(projector(states["|0⟩"]), projector(normalize([1.0, -1.0])))



0.4999999999999999

In [194]:
ket0 = normalize([-1.0, 0.55])
ρ_true = projector(ket0)

probabilities_true = [predicted_probability(E, ρ_true) for E in measurements]

4-element Vector{Float64}:
 0.7677543186180422
 0.23224568138195778
 0.07773512476007674
 0.922264875239923

In [193]:
measurement_noise = [0.06, -0.03, 0.08, -0.02]
probabilities_observed = probabilities_true .+ measurement_noise

collect(zip(measurement_names, probabilities_true, probabilities_observed))

4-element Vector{Tuple{String, Float64, Float64}}:
 ("|0⟩", 0.7677543186180422, 0.8277543186180423)
 ("|1⟩", 0.23224568138195778, 0.20224568138195778)
 ("|+⟩", 0.07773512476007674, 0.15773512476007673)
 ("|−⟩", 0.922264875239923, 0.9022648752399229)

## Core helper functions

The three optimization models below share the same building blocks

Implementing the helpers

`real_density_variables`
`residual_vector`
`add_least_squares_objective!`


 hopefully makes the optimization structure easier to see.

In [154]:
function real_density_variables(model)
    @variable(model, ρ11)
    @variable(model, ρ12)
    @variable(model, ρ22)

    return [
        ρ11  ρ12
        ρ12  ρ22
    ]
end

function residual_vector(probabilities, measurements, ρ)
    residuals = probabilities.- predicted_probability.(measurements, Ref(ρ))
end

function add_least_squares_objective!(model, residuals)
    @objective(model, Min, sum(residuals.^2))
    return model
end


add_least_squares_objective! (generic function with 1 method)

### Version 1: Tomography as unconstrained least squares

First things first, just fit the observations. It might fit the observations well, but it doesn't have to be a valid density matrix.

In [155]:
function tomography_model_v1(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)

    set_silent(model)
    optimize!(model)

    return (
        ρ=value.(ρ), 
        objective=objective_value(model), 
        status=termination_status(model)
    )
end

tomography_model_v1 (generic function with 1 method)

In [192]:
ρ_v1, J_v1, status_v1 = tomography_model_v1(probabilities_observed, measurements)
eigvals(ρ_v1)

2-element Vector{Float64}:
 0.03629386968937606
 1.0087061303106237

### Version 2: add the trace constraint

A density matrix needs a constraint on its trace!  JuMP and Ipopt solve the resulting constrained optimization problem for us, but we still need to tell JuMP what the constraint is.




In [160]:
function add_trace_constraint!(model, ρ)

    @constraint(model, tr(ρ) == 1)
    
    return model
end

function tomography_model_v2(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)
    add_trace_constraint!(model, ρ)

    set_silent(model)
    optimize!(model)

    return (
        ρ=value.(ρ), 
        objective=objective_value(model), 
        status=termination_status(model)
    )
end


tomography_model_v2 (generic function with 1 method)

In [161]:
ρ_v2, J_v2, status_v2 = tomography_model_v2(probabilities_observed, measurements)
ρ_v2


2×2 Matrix{Float64}:
  0.812754  -0.372265
 -0.372265   0.187246

### Version 3: add the positive-semidefinite constraint

Let's solve the physically constrained tomography problem, in all its glory.

The density matrix needs positive eigenvalues. This one turns out to be nonlinear!

In [162]:
function add_real_2x2_psd_constraints!(model, ρ)
    #THIS DOES NOT WORK FOR SOME WEIRD REASON
    a = ρ[1,1]
    b = ρ[1,2]
    d = ρ[2,2]
    @constraint(model, a >= 0)
    @constraint(model, d >= 0)
    @constraint(model, a*d-b^2 >= 0)
    return model
end

function tomography_model_v3(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)
    add_trace_constraint!(model, ρ)
    add_real_2x2_psd_constraints!(model, ρ)

    set_silent(model)
    optimize!(model)

    return value.(ρ), objective_value(model), termination_status(model)
end


tomography_model_v3 (generic function with 1 method)

In [164]:
ρ_v3, J_v3, status_v3 = tomography_model_v3(probabilities_observed, measurements)
ρ_v3


2×2 Matrix{Float64}:
  0.812754  -0.372265
 -0.372265   0.187246

### Diagnostics: physicality, reconstruction error, and Bloch vectors

A tomography estimate can have a small least-squares objective and still fail to be a physical density matrix. To diagnose this, compare:

- trace: should be close to `1`;
- minimum eigenvalue: should be nonnegative;
- reconstruction error: distance from the known simulated state `ρ_true`;
- Bloch vector: a geometric representation of the qubit state.

For the real matrices in this core notebook, the Bloch vector has no $y$ component:

$$
\rho = \frac{1}{2}
\begin{bmatrix}
1+z & x \\
x & 1-z
\end{bmatrix},
\qquad
x = 2\rho_{12},\quad y=0,\quad z=\rho_{11}-\rho_{22}.
$$

A physical state should lie inside the Bloch sphere. Pure states lie on the surface.


In [191]:
function is_physical_density_matrix(ρ; atol = 1e-8)
    return isapprox(tr(ρ), 1, atol= atol) && minimum(eigvals(ρ))>= -1 * atol
end

function reconstruction_error(ρ, ρ_reference)
    return tr(transpose(ρ-ρ_reference)(ρ-ρ_reference))^1/2
end

function bloch_vector(ρ)
    return [2*ρ[1,2],0,(ρ[1,1] - ρ[2,2])]
end

reconstruction_records = [
    (name = "true state", ρ = ρ_true, objective = 0.0, status = missing),
    (name = "v1: unconstrained LS", ρ = ρ_v1, objective = J_v1, status = status_v1),
    (name = "v2: trace constrained", ρ = ρ_v2, objective = J_v2, status = status_v2),
    (name = "v3: trace + PSD constrained", ρ = ρ_v3, objective = J_v3, status = status_v3),
]

data = DataFrame(reconstruction_records)


Row,name,ρ,objective,status
,String,Array…,Float64,Terminat…?
1,true state,[0.767754 -0.422265; -0.422265 0.232246],0.0,missing
2,v1: unconstrained LS,[0.835254 -0.372265; -0.372265 0.209746],0.000225,LOCALLY_SOLVED
3,v2: trace constrained,[0.812754 -0.372265; -0.372265 0.187246],0.00225,LOCALLY_SOLVED
4,v3: trace + PSD constrained,[0.812754 -0.372265; -0.372265 0.187246],0.00225,LOCALLY_SOLVED


In [190]:
function plot_bloch_vectors(reconstruction_records)
    b = Bloch()
    fig, lscene = render(b)
    clear!(b)
    for i in range(1,4)
        add_vectors!(b, bloch_vector(reconstruction_records[i].ρ))
    end
    fig, _ = render(b)
    fig
    # TODO: Plot Bloch-sphere vectors for the true state and the three reconstructions.
end

plot_bloch_vectors(reconstruction_records)


---

## Extension projects

The core notebook reconstructs a real two-level density matrix from four projective measurements with fixed additive noise. The projects below each change one modeling assumption. They are designed to connect the same optimization framework to experimental design, complex quantum states, statistical noise models, and explicit linear-algebra solutions.


### Project 1: remove measurements and study identifiability

**What you will do.** Repeat the reconstruction using only subsets of the measurements. For example, try the $Z$ basis only, the $X$ basis only, or one measurement from each basis.

**Why it matters.** Tomography is only possible when the measurements contain enough information to determine the unknown state. If measurements are removed, the optimization problem can become underdetermined: many density matrices may explain the same data. The trace and PSD constraints can reduce the set of possible answers, but they cannot create information that was never measured.

**Questions to answer.**

- Which subsets give a unique-looking reconstruction?
- Which subsets allow many states to fit the data equally well?
- When measurements are missing, do the trace and PSD constraints help or hide the lack of information?
- Can two different physical density matrices produce the same observed probabilities for a chosen subset?


In [ ]:
measurement_subsets = Dict(
    "Z basis only" => ["|0⟩", "|1⟩"],
    "X basis only" => ["|+⟩", "|−⟩"],
    "one Z and one X" => ["|0⟩", "|+⟩"],
    "missing |−⟩" => ["|0⟩", "|1⟩", "|+⟩"],
)

function make_measurements(names)
    return [projector(states[name]) for name in names]
end

subset_name = "Z basis only"
measurements_subset = make_measurements(measurement_subsets[subset_name])
probabilities_true_subset = [predicted_probability(E, ρ_true) for E in measurements_subset]

# TODO: Decide how to construct observed probabilities for the subset.
# Option 1: use the noiseless probabilities_true_subset.
# Option 2: add a subset of measurement_noise.
# Option 3: simulate finite-shot data after completing Project 3.
probabilities_observed_subset = error("TODO: Choose observed data for the measurement subset")

# TODO: Run tomography_model_v1, tomography_model_v2, and tomography_model_v3 on the subset.
# TODO: Compare objectives, physicality, and Bloch vectors across measurement subsets.
identifiability_results = error("TODO: Study identifiability across measurement subsets")


### Project 2: add Y-basis measurements and complex Hermitian density matrices

**What you will do.** Generalize the notebook from real density matrices to complex Hermitian density matrices by adding the Pauli-$Y$ basis measurements:

$$
|+i\rangle = \frac{1}{\sqrt{2}}\begin{bmatrix}1 \\ i\end{bmatrix},
\qquad
|-i\rangle = \frac{1}{\sqrt{2}}\begin{bmatrix}1 \\ -i\end{bmatrix}.
$$

The density matrix becomes

$$
\rho =
\begin{bmatrix}
\rho_{11} & a + ib \\
a - ib & \rho_{22}
\end{bmatrix}.
$$

**Why it matters.** The real core notebook can only represent states with Bloch vectors in the $x$-$z$ plane. A general qubit also has a $y$ component, which appears as imaginary coherence in the off-diagonal entries. Without $Y$-basis measurements, that imaginary part is not identifiable.

**Questions to answer.**

- What changes in the Born-rule prediction when measurement matrices are complex?
- How does the PSD determinant constraint change?
- Can the original $X/Z$ measurements recover a state with nonzero imaginary coherence?
- What happens to the Bloch vectors after adding the $Y$-basis measurements?


In [ ]:
states_complex = Dict(
    "|0⟩" => ComplexF64[1.0, 0.0],
    "|1⟩" => ComplexF64[0.0, 1.0],
    "|+⟩" => normalize(ComplexF64[1.0, 1.0]),
    "|−⟩" => normalize(ComplexF64[1.0, -1.0]),
    "|+i⟩" => normalize(ComplexF64[1.0, 1.0im]),
    "|−i⟩" => normalize(ComplexF64[1.0, -1.0im]),
)

projector_complex(ψ) = ψ * ψ'

measurement_names_complex = ["|0⟩", "|1⟩", "|+⟩", "|−⟩", "|+i⟩", "|−i⟩"]
measurements_complex = [projector_complex(states_complex[name]) for name in measurement_names_complex]

ketφ = normalize(ComplexF64[1.0, 0.4 + 0.7im])
ρ_true_complex = projector_complex(ketφ)
probabilities_true_complex = [real(tr(E * ρ_true_complex)) for E in measurements_complex]

function tomography_model_complex(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    @variable(model, ρ11)
    @variable(model, ρ22)
    @variable(model, ρ12_re)
    @variable(model, ρ12_im)

    # TODO: Build predicted probabilities for the complex Hermitian matrix.
    # Hint: if E is Hermitian and ρ[1, 2] = ρ12_re + im * ρ12_im, then
    #   tr(Eρ) = real(E[1,1])ρ11 + real(E[2,2])ρ22
    #            + 2real(E[1,2])ρ12_re + 2imag(E[1,2])ρ12_im.
    predicted = error("TODO: Implement complex Born-rule predictions")

    residuals = predicted .- probabilities

    # TODO: Add least-squares or MLE objective.
    # TODO: Add trace constraint ρ11 + ρ22 == 1.
    # TODO: Add PSD constraints:
    #       ρ11 ≥ 0, ρ22 ≥ 0,
    #       ρ11 * ρ22 - ρ12_re^2 - ρ12_im^2 ≥ 0.
    error("TODO: Complete tomography_model_complex")

    set_silent(model)
    optimize!(model)

    ρ_estimate = [
        value(ρ11)               value(ρ12_re) + im * value(ρ12_im)
        value(ρ12_re) - im * value(ρ12_im)   value(ρ22)
    ]

    return ρ_estimate, objective_value(model), termination_status(model)
end

# TODO: Reconstruct ρ_true_complex from probabilities_true_complex or noisy observations.
ρ_complex, J_complex, status_complex = error("TODO: Run complex tomography")


### Project 3: simulate finite-shot count noise and use MLE / cross-entropy

**What you will do.** Replace fixed additive probability noise with simulated measurement counts. Then compare least squares on empirical frequencies with maximum likelihood estimation on the counts.

For a yes/no measurement with $N_i$ shots and $k_i$ observed successes,

$$
k_i \sim \mathrm{Binomial}(N_i, q_i),
\qquad
q_i = \mathrm{tr}(E_i\rho).
$$

The negative log-likelihood, ignoring constants that do not depend on $\rho$, is

$$
-\sum_i \left[k_i\log(q_i) + (N_i-k_i)\log(1-q_i)\right].
$$

This is the binary cross-entropy loss.

**Why it matters.** Least squares treats all probability errors symmetrically and does not know how many shots produced each estimate. MLE uses the count model directly and usually behaves better when probabilities are near 0 or 1, or when different measurements have different shot counts.

**Questions to answer.**

- How do least squares and MLE compare when the number of shots is small?
- How many shots are needed before the two reconstructions look similar?
- What numerical safeguards are needed when $q_i$ is close to `0` or `1`?
- How does the PSD constraint interact with noisy count data?


In [ ]:
function binomial_count(shots, p)
    return count(rand() < p for _ in 1:shots)
end

Random.seed!(1234)
shots = fill(100, length(probabilities_true))
counts = [binomial_count(shots[i], probabilities_true[i]) for i in eachindex(shots)]
frequencies = counts ./ shots

collect(zip(measurement_names, counts, shots, frequencies))


In [ ]:
function tomography_model_mle(counts, shots, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    q = [predicted_probability(E, ρ) for E in measurements]

    add_trace_constraint!(model, ρ)
    add_real_2x2_psd_constraints!(model, ρ)

    ϵ = 1e-6

    # TODO: Constrain every predicted probability q[i] to stay in [ϵ, 1 - ϵ].
    # This prevents log(0) in the likelihood.
    error("TODO: Add probability-domain constraints")

    # TODO: Add the negative binomial log-likelihood / binary cross-entropy objective:
    #   -sum(counts[i] * log(q[i]) + (shots[i] - counts[i]) * log(1 - q[i]) for i in eachindex(counts))
    error("TODO: Add MLE / cross-entropy objective")

    set_silent(model)
    optimize!(model)

    return value.(ρ), objective_value(model), termination_status(model)
end

# TODO: Compare least squares on frequencies with MLE on counts.
ρ_ls_counts, J_ls_counts, status_ls_counts = error("TODO: Run least-squares tomography on frequencies")
ρ_mle, J_mle, status_mle = error("TODO: Run tomography_model_mle")


### Project 4: solve the trace-constrained model by explicit KKT

**What you will do.** Derive and solve the trace-constrained least-squares problem directly as a linear algebra system, then compare the result to `tomography_model_v2`.

For a real symmetric matrix, write the unknowns as

$$
x =
\begin{bmatrix}
\rho_{11} \\
\rho_{12} \\
\rho_{22}
\end{bmatrix}.
$$

Each measurement gives one row of a design matrix $A$:

$$
\mathrm{tr}(E_i\rho)
=
E_{i,11}\rho_{11}
+
(E_{i,12}+E_{i,21})\rho_{12}
+
E_{i,22}\rho_{22}.
$$

The trace constraint (remember, trace is linear, so it has a matrix!) is

$$
c^\top x = 1,
\qquad
c = \begin{bmatrix}1 \\ 0 \\ 1\end{bmatrix}.
$$

The KKT system is

$$
\begin{bmatrix}
2A^\top A & c \\
c^\top & 0
\end{bmatrix}
\begin{bmatrix}
x \\
\lambda
\end{bmatrix}
=
\begin{bmatrix}
2A^\top \hat{p} \\
1
\end{bmatrix}.
$$

**Why it matters.** This project shows what the optimization solver is doing in one special case. It connects JuMP modeling back to linear algebra, normal equations, and Lagrange multipliers.

**Questions to answer.**

- Does the KKT solution match `tomography_model_v2`?
- What happens when measurements are removed and $A$ loses rank?
- Why does this explicit KKT method not directly handle the PSD constraint from Version 3?


In [ ]:
function design_matrix_real(measurements)
    # TODO: Build the matrix A whose rows map
    # x = [ρ11, ρ12, ρ22] to predicted probabilities.
    error("TODO: Implement design_matrix_real")
end

function solve_trace_constrained_kkt(probabilities, measurements)
    A = design_matrix_real(measurements)
    c = [1.0, 0.0, 1.0]

    # TODO: Build and solve the KKT system.
    # Hint: use block matrices and the backslash operator.
    solution = error("TODO: Solve the KKT system")

    x = solution[1:3]
    λ = solution[4]

    ρ = [
        x[1]  x[2]
        x[2]  x[3]
    ]

    return ρ, λ
end

ρ_kkt, λ_kkt = solve_trace_constrained_kkt(probabilities_observed, measurements)

# TODO: Compare ρ_kkt with ρ_v2.
comparison_to_jump = error("TODO: Compare KKT solution to tomography_model_v2")
